# Dropout e batch normalization



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Aggiungerai dropout al modello Spotify e vedrai come batch normalization può facilitare il training su dati difficili. Esegui la cella iniziale.


In [ ]:
# Configura i grafici.
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('animation', html='html5')

# Configura il sistema di verifica degli esercizi.
from learntools.core import binder
binder.bind(globals())
from learntools.deep_learning_intro.ex5 import *

Carica il dataset **Spotify**.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import GroupShuffleSplit

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import callbacks

spotify = pd.read_csv('../input/dl-course-data/spotify.csv')

X = spotify.copy().dropna()
y = X.pop('track_popularity')
artists = X['track_artist']

features_num = ['danceability', 'energy', 'key', 'loudness', 'mode',
                'speechiness', 'acousticness', 'instrumentalness',
                'liveness', 'valence', 'tempo', 'duration_ms']
features_cat = ['playlist_genre']

preprocessor = make_column_transformer(
    (StandardScaler(), features_num),
    (OneHotEncoder(), features_cat),
)

def group_split(X, y, group, train_size=0.75):
    splitter = GroupShuffleSplit(train_size=train_size)
    train, test = next(splitter.split(X, y, groups=group))
    return (X.iloc[train], X.iloc[test], y.iloc[train], y.iloc[test])

X_train, X_valid, y_train, y_valid = group_split(X, y, artists)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
y_train = y_train / 100
y_valid = y_valid / 100

input_shape = [X_train.shape[1]]
print("Input shape: {}".format(input_shape))

# 1. Aggiungere dropout
Parti dal modello dell'esercizio 4. Inserisci un layer **Dropout** dopo Dense da **128 unità** e un altro dopo Dense da **64 unità**. Imposta **`rate=0.3`** in entrambi.


In [ ]:
# SCRIVI QUI IL CODICE: aggiungi due layer di dropout al 30%, dopo i layer da 128 e 64 unità.
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=input_shape),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# Controlla la risposta.
q_1.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_1.hint()
#q_1.solution()

Esegui il training per osservare l'effetto del dropout.


In [ ]:
model.compile(
    optimizer='adam',
    loss='mae',
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=50,
    verbose=0,
)
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()
print("Minimum Validation Loss: {:0.4f}".format(history_df['val_loss'].min()))

# 2. Valutare il dropout
Nell'esercizio 4 il modello tendeva a sovradattarsi intorno all'epoca 5. Il dropout sembra aver limitato il problema?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_2.check()

Ora studiamo la batch normalization. Carica **Concrete** senza standardizzare: le differenze di scala renderanno più evidente l'effetto.


In [ ]:
import pandas as pd

concrete = pd.read_csv('../input/dl-course-data/concrete.csv')
df = concrete.copy()

df_train = df.sample(frac=0.7, random_state=0)
df_valid = df.drop(df_train.index)

X_train = df_train.drop('CompressiveStrength', axis=1)
X_valid = df_valid.drop('CompressiveStrength', axis=1)
y_train = df_train['CompressiveStrength']
y_valid = df_valid['CompressiveStrength']

input_shape = [X_train.shape[1]]

Addestra la rete sui dati Concrete non standardizzati.


In [ ]:
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=input_shape),
    layers.Dense(512, activation='relu'),    
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])
model.compile(
    optimizer='sgd', # SGD è più sensibile alle differenze di scala.
    loss='mae',
    metrics=['mae'],
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=64,
    epochs=100,
    verbose=0,
)

history_df = pd.DataFrame(history.history)
history_df.loc[0:, ['loss', 'val_loss']].plot()
print(("Minimum Validation Loss: {:0.4f}").format(history_df['val_loss'].min()))

Il grafico è vuoto o la loss molto alta? Nell'esempio questa configurazione spesso non converge; anche un'inizializzazione fortunata può dare risultati scarsi.

# 3. Aggiungere batch normalization
Inserisci **quattro layer BatchNormalization**, uno prima di ogni Dense. Sposta la forma dell'ingresso sul nuovo primo layer.


In [ ]:
# SCRIVI QUI IL CODICE: aggiungi un layer BatchNormalization prima di ogni layer Dense.
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=input_shape),
    layers.Dense(512, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(1),
])

# Controlla la risposta.
q_3.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_3.hint()
#q_3.solution()

Esegui la cella per verificare se ora il modello riesce ad addestrarsi.


In [ ]:
model.compile(
    optimizer='sgd',
    loss='mae',
    metrics=['mae'],
)
EPOCHS = 100
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=64,
    epochs=EPOCHS,
    verbose=0,
)

history_df = pd.DataFrame(history.history)
history_df.loc[0:, ['loss', 'val_loss']].plot()
print(("Minimum Validation Loss: {:0.4f}").format(history_df['val_loss'].min()))

# 4. Valutare batch normalization
Ha migliorato l'addestramento? Confronta i risultati e motiva la risposta.


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_4.check()

Prosegui con le **[reti per la classificazione binaria](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/tut6.html)**.
